# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet + GPU trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Clone repo của bạn (cần đã push). Nếu thư mục đã có thì bỏ qua.
REPO_URL = "https://github.com/long2110d/K4-DAY02-DoDinhLong-2A202602673.git"
REPO_DIR = "K4-DAY02-DoDinhLong-2A202602673"
CODE_DIR = f"{REPO_DIR}/submissions/2A202602673_do_dinh_long/code"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)    # để import eval.py (bản gốc, không sửa)
sys.path.insert(0, CODE_DIR)    # dataset.py, model.py, losses.py, train.py, inference.py, benchmark.py

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
# Tự tìm thư mục chứa các file .jpg sau khi giải nén (data/images hoặc data/ ...).
import glob
_probe = "20160928-140314-0.jpg"
_hits = glob.glob(f"data/**/{_probe}", recursive=True)
assert _hits, "không thấy ảnh sau khi giải nén, hãy `ls data`"
IMAGES_DIR = os.path.dirname(_hits[0])
LABELS_DIR = "data/labels"
print("IMAGES_DIR =", IMAGES_DIR)

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import dataset as ds
import matplotlib.pyplot as plt
from PIL import Image

train_df, val_df, test_df = ds.load_split(LABELS_DIR, fold=0)
info = ds.check_split(train_df, val_df, test_df, IMAGES_DIR)   # assert nếu vi phạm S1-S6

dist = pd.DataFrame(info["per_class"], index=ds.CLASS_NAMES)
display(dist.assign(total=dist.sum(1)))
dist.plot.bar(figsize=(9, 4), title="Số ảnh mỗi lớp theo tập (fold 0)")
plt.tight_layout(); plt.show()

# >= 3 ảnh mỗi lớp (từ tập train)
fig, ax = plt.subplots(9, 3, figsize=(7, 21))
for c in range(9):
    for j, fn in enumerate(train_df[train_df.Label == c].Filename.head(3)):
        ax[c, j].imshow(Image.open(f"{IMAGES_DIR}/{fn}")); ax[c, j].axis("off")
        if j == 0: ax[c, j].set_title(ds.CLASS_NAMES[c], fontsize=9, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
import math, torch, torch.nn as nn
import model as mdl, losses, train
from train import Config

train.set_seed(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tl = ds.make_loader(train_df, IMAGES_DIR, ds.build_transforms(True), 32, True, None, 2)
xb, yb, fb = next(iter(tl))

# 2) loss ban đầu của head mới ~ -ln(1/9) = 2.197
m = mdl.build_model("resnet50", True, 9).to(dev)
m.eval()
with torch.no_grad():
    print("loss khởi đầu:", nn.functional.cross_entropy(m(xb.to(dev)), yb.to(dev)).item(), "| kỳ vọng ~", math.log(9))

# 3) overfit một batch nhỏ tới loss gần 0
m.train()
opt = torch.optim.AdamW(mdl.param_groups(m, 1e-4, 1e-3, 0.0))
x8, y8 = xb[:8].to(dev), yb[:8].to(dev)
for step in range(60):
    opt.zero_grad(); loss = nn.functional.cross_entropy(m(x8), y8); loss.backward(); opt.step()
print("loss sau 60 bước trên 1 batch nhỏ:", loss.item())

# 4) vẽ ảnh sau augmentation (giải chuẩn hoá) cùng nhãn
mean = torch.tensor(ds.IMAGENET_MEAN).view(3, 1, 1); std = torch.tensor(ds.IMAGENET_STD).view(3, 1, 1)
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for a, x, y, f in zip(ax.ravel(), xb, yb, fb):
    a.imshow((x * std + mean).clamp(0, 1).permute(1, 2, 0)); a.set_title(f"{ds.CLASS_NAMES[y]}", fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()
del m, opt

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from train import Config, run
import time, json

BACKBONES = ["resnet50", "resnext50_32x4d", "convnext_tiny", "vit_small_patch16_224", "efficientnet_b0"]
# Nếu batch 64 bị OOM trên Kaggle, giảm BATCH (ghi rõ trong báo cáo).
BATCH = 64
EPOCHS = 12
common = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, batch_size=BATCH, epochs=EPOCHS)

rows = []
for i, bb in enumerate(BACKBONES, 1):
    s = run(Config(exp_id=f"B{i:02d}", backbone=bb, seed=0, **common))   # val-only: save_test_predictions=False
    rows.append(s)
    pd.DataFrame(rows).to_csv("backbones.csv", index=False)
bb_table = pd.DataFrame(rows)[["exp_id", "backbone", "weight_tag", "params_m", "gmacs", "val_macro_f1", "val_top1", "sec_per_epoch", "best_epoch"]]
display(bb_table)
# TODO (bạn): chọn 1-2 backbone đi tiếp, ghi lý do dựa trên bảng trên.
BEST_BACKBONE = bb_table.sort_values("val_macro_f1", ascending=False).iloc[0].backbone
print("backbone có macro-F1 val cao nhất:", BEST_BACKBONE)

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# Mỗi thí nghiệm chỉ khác T00 MỘT yếu tố (T00 = công thức nền, CE, aug basic, finetune, không EMA).
base = dict(backbone=BEST_BACKBONE, seed=0, **common)
exps = {
    "T00": {},                                                  # nền
    "T01": dict(loss="ls", label_smoothing=0.1),                # trục C: label smoothing
    "T02": dict(loss="ce_weighted", class_weight_beta=0.0),     # trục C: CE theo trọng số lớp (1/n_c)
    "T03": dict(sampler="balanced"),                            # trục D: sampler cân bằng lớp
    "T04": dict(aug="trivial"),                                 # trục B: augmentation mạnh hơn
    "T05": dict(mix="cutmix", mix_alpha=1.0),                   # trục B: CutMix
    "T06": dict(ema_decay=0.999),                               # trục F: EMA
}
rows = []
for eid, delta in exps.items():
    s = run(Config(exp_id=eid, **{**base, **delta}))
    s["delta"] = json.dumps(delta); rows.append(s)
    pd.DataFrame(rows).to_csv("training.csv", index=False)
train_table = pd.DataFrame(rows)[["exp_id", "delta", "val_macro_f1", "val_top1", "val_nll", "best_epoch", "sec_per_epoch"]]
train_table["d_f1_vs_T00"] = train_table.val_macro_f1 - train_table.val_macro_f1.iloc[0]
display(train_table)
# TODO (bạn): so Δ với nhiễu (chạy T00 thêm vài seed để ước lượng std), rồi thử một kết hợp tốt:
# rows.append(run(Config(exp_id="T07", **{**base, **dict(loss="ls", label_smoothing=0.1, ema_decay=0.999)})))

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
import torch, numpy as np
import inference as inf, benchmark as bench
from train import run_dir
import eval as ev

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INF_EXP = "T00"   # chọn run (seed 0) đã huấn luyện ở Bước 2 để làm suy luận; đổi theo kết quả val của bạn
cfg = Config(exp_id=INF_EXP, backbone=BEST_BACKBONE, seed=0, **common)
net = mdl.build_model(cfg.backbone, pretrained=False, num_classes=9).to(dev)
net.load_state_dict(torch.load(run_dir(cfg) / "best.pt", map_location=dev))
val_loader = ds.make_loader(val_df, IMAGES_DIR, ds.build_transforms(False), 128, False, num_workers=2)

def val_scores(p, y):
    m = ev.compute_metrics(y, p.argmax(1), p)
    return dict(macro_f1=m["macro_f1"], top1=m["top1"], nll=m["nll"], ece=m["ece"])

res = {}
_, yv, z_id = inf.predict_logits(net, val_loader, dev)                                   # I00: 1 view
_, _, z_fl = inf.predict_logits(net, val_loader, dev, inf.view_hflip)
res["I00 1-view"] = val_scores(inf.aggregate_views([z_id], "prob"), yv)
res["I01 hflip, prob"] = val_scores(inf.aggregate_views([z_id, z_fl], "prob"), yv)
res["I02 hflip, logit"] = val_scores(inf.aggregate_views([z_id, z_fl], "logit"), yv)
# I03: multi-crop 5 view (crop 224 trên ảnh 256 chưa crop) cần loader 256: dùng transform riêng
tf256 = ds.build_transforms(False); tf256.transforms[1] = __import__("torchvision").transforms.CenterCrop(256)
l256 = ds.make_loader(val_df, IMAGES_DIR, tf256, 64, False, num_workers=2)
zs = [inf.predict_logits(net, l256, dev, lambda x, i=i: inf.views_multicrop(x, 224, True)[i])[2] for i in range(10)]
res["I03 multicrop 5+flip, prob"] = val_scores(inf.aggregate_views(zs, "prob"), yv)
# I04: temperature scaling (T khớp trên VAL; ở đây đánh giá ngay trên val chỉ để xem ECE/NLL)
T = inf.fit_temperature(z_id, yv); print("T =", T)
res["I04 1-view + temperature"] = val_scores(inf.apply_temperature(z_id, T), yv)
display(pd.DataFrame(res).T)

# Độ trễ (forward model, batch 1, KHÔNG gồm tiền xử lý)
lat = [bench.latency_report(net, 1, 224, d, "cuda" if torch.cuda.is_available() else "cpu", warmup=20, iters=200) for d in ("fp32", "amp", "fp16")]
lat.append({**bench.latency_report(inf.fuse_conv_bn(net) if BEST_BACKBONE in ("resnet50", "resnext50_32x4d") else net, 1, 224, "fp32", "cuda" if torch.cuda.is_available() else "cpu", warmup=20, iters=200), "note": "BN fused (nếu có)"})
lat.append({**bench.tta_latency(net, 2, img_size=224, dtype="fp32", device="cuda" if torch.cuda.is_available() else "cpu"), "note": "TTA 2 view"})
lat_table = pd.DataFrame(lat); display(lat_table)

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
import eval as ev, torch.nn.functional as Fn

SEEDS = (0, 1, 2)
# TODO (bạn): đặt cấu hình cuối theo kết quả VAL ở Bước 1-3, ví dụ:
FINAL = dict(backbone=BEST_BACKBONE, loss="ls", label_smoothing=0.1, ema_decay=0.999)
INF = dict(views=("id", "hflip"), space="prob", temperature=True)   # suy luận cuối (chọn trên val)

os.makedirs("predictions", exist_ok=True)
test_loader = ds.make_loader(test_df, IMAGES_DIR, ds.build_transforms(False), 128, False, num_workers=2)
for seed in SEEDS:
    # mốc: T00 + I00, test ghi bởi run() (đúng MỘT lần)
    run(Config(exp_id="T00", seed=seed, **{**common, "backbone": BEST_BACKBONE}, save_test_predictions=True))
    # final: huấn luyện KHÔNG ghi test; suy luận cuối (TTA + temperature) chạy test đúng một lần ở dưới
    fcfg = Config(exp_id="F01", seed=seed, **{**common, **FINAL})
    run(fcfg)
    net = mdl.build_model(fcfg.backbone, False, 9).to(dev)
    net.load_state_dict(torch.load(run_dir(fcfg) / "best.pt", map_location=dev))
    vviews = [inf.predict_logits(net, val_loader, dev, v)[2] for v in (inf.view_identity, inf.view_hflip)]
    tviews = [inf.predict_logits(net, test_loader, dev, v) for v in (inf.view_identity, inf.view_hflip)]
    names, yt = tviews[0][0], tviews[0][1]
    z_val, z_test = np.mean(vviews, 0), np.mean([t[2] for t in tviews], 0)    # trung bình logit hai view
    _, yv, _ = inf.predict_logits(net, val_loader, dev)
    Tcal = inf.fit_temperature(z_val, yv)                                      # T khớp trên VAL
    ev.save_predictions(f"predictions/F01uncal_seed{seed}_test.csv", names, yt, inf.apply_temperature(z_test, 1.0))
    ev.save_predictions(f"predictions/F01_seed{seed}_test.csv", names, yt, inf.apply_temperature(z_test, Tcal))
    print(f"seed {seed}: T = {Tcal:.3f}")

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
import glob
from train import Config
with pd.ExcelWriter("results.xlsx", engine="openpyxl") as xw:
    for name, df in [("Backbones", bb_table), ("Training", train_table), ("Inference", pd.DataFrame(res).T.reset_index()),
                     ("Latency", lat_table)]:
        df.to_excel(xw, sheet_name=name, index=False)
    # Final/PerClass/Summary: dán số từ eval_out/ (do eval.py sinh ra) để khớp đúng định nghĩa chấm
    for f in sorted(glob.glob("eval_out/*.csv")):
        pd.read_csv(f).to_excel(xw, sheet_name=os.path.basename(f)[:31].replace(".csv", ""), index=False)
print("curves có:", sorted(os.listdir("curves")))
# TODO (bạn): viết report.md; kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp eval.py.